# PCR — Priority-Conditioned Retention (test-time, Mamba-790m)
**Importance-Aware Capacity Allocation in a *pretrained* continuous-state SSM**

이 노트북은 *학습 없이* 사전학습 Mamba-790m에서 retention을 추론 시점에 제어한다.
원래 제안서의 "학습된 [P1/P2/P3] grammar + d_state 용량법칙 + 3-way"가 아니라,
**실제로 검증된 test-time 기계해석 프로그램**을 처음부터 재현한다.

- 솔직한 스코프: **790m 단일 모델**. 1.4b 비재현·자연어 fragile은 limitation.
- 두 레버: (1) 생존 = Δ 게이트, (2) 정밀도 = write 양자화.
- T4에서 수십 분. `mamba-ssm`/`causal-conv1d`는 **설치하지 않는다**(slow path 강제 → Δ 훅 가능).

In [ ]:
# [설치] fast path 막아서 slow path 강제 → dt_proj/conv1d 훅 가능
!pip -q install "transformers>=4.39" accelerate
import torch, random, math
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
# [로드] 사전학습 790m, fp32, freeze
from transformers import AutoTokenizer, MambaForCausalLM
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL = "state-spaces/mamba-790m-hf"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = MambaForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32).to(DEVICE).eval()
for p in model.parameters(): p.requires_grad_(False)
NL = len(model.backbone.layers)
print("layers", NL)
print("주의: 'fast path is not available ... sequential implementation' 경고는 정상 = slow path 확인")

In [ ]:
# [Δ 훅] dt_proj 출력(pre-softplus)에 위치별 gain. _CTRL로 제어.
_CTRL = {'gain_vec': None, 'layers': None}
def _mk_dt_hook(idx):
    def hook(mod, inp, out):
        g = _CTRL['gain_vec']
        if g is None: return out
        if _CTRL['layers'] is not None and idx not in _CTRL['layers']: return out
        L = out.shape[1]
        gv = g[:L].to(out.dtype).to(out.device)
        return out * gv.view(1, L, 1)
    return hook
_dt_handles = []
def setup_dt_hooks():
    global _dt_handles
    for h in _dt_handles: h.remove()
    _dt_handles = [l.mixer.dt_proj.register_forward_hook(_mk_dt_hook(i))
                   for i, l in enumerate(model.backbone.layers)]
def verify_dt_hooks():
    ids = tokenizer(" a b c d e f", return_tensors='pt').input_ids.to(DEVICE)
    with torch.no_grad(): base = model(ids).logits[0, -1].clone()
    _CTRL['gain_vec'] = torch.full((ids.shape[1],), 0.5); _CTRL['layers'] = None
    with torch.no_grad(): pert = model(ids).logits[0, -1]
    _CTRL['gain_vec'] = None
    d = (base - pert).abs().max().item()
    return d > 1e-4, d
setup_dt_hooks()
ok, d = verify_dt_hooks(); print("Δ hook fires:", ok, "| max logit change", round(d, 3))

In [ ]:
# [진단] 레이어별 평균 decay A (-exp(A_log)). A<0 → recency 편향.
_A = [(-torch.exp(l.mixer.A_log)).mean().item() for l in model.backbone.layers]
print("mean A: L0", round(_A[0],3), "| Lmid", round(_A[NL//2],3), "| Llast", round(_A[-1],3))

In [ ]:
# [데이터] single-token 단어 풀 + MQAR (각 단어=1토큰 → 위치 정확)
def single_token_words(n, seed=0):
    rng = random.Random(seed); ids = list(range(1000, 45000)); rng.shuffle(ids); out = []
    for i in ids:
        s = tokenizer.decode([i]).strip()
        if (s.isascii() and s.isalpha() and 3 <= len(s) <= 8
                and len(tokenizer(" " + s, add_special_tokens=False).input_ids) == 1):
            out.append(s)
        if len(out) >= n: break
    return out
_WORDS = single_token_words(4000); print("single-token words:", len(_WORDS))

def _enc(word):
    return tokenizer(" " + word, add_special_tokens=False).input_ids

def build_mqar(N, rng, query_pair=0):
    ws = rng.sample(_WORDS, 2 * N)
    pairs = [(ws[2*j], ws[2*j+1]) for j in range(N)]
    seq = []; pos = []
    for (k, v) in pairs:
        kp = len(seq); seq += _enc(k)
        vp = len(seq); seq += _enc(v); pos.append((kp, vp))
    seq += tokenizer(" ?", add_special_tokens=False).input_ids
    seq += _enc(pairs[query_pair][0])
    ids = torch.tensor([seq], device=DEVICE)
    tgt = _enc(pairs[query_pair][1])[0]
    return ids, pos, tgt, pairs

def _recall_once(ids, tgt):
    with torch.no_grad(): lg = model(ids).logits[0, -1]
    return int(lg.argmax().item() == tgt)

In [ ]:
# [용량 곡선] base recall vs N → N* 찾기
def base_recall(N, n=30, seed0=0):
    h = 0
    for t in range(n):
        ids, pos, tgt, _ = build_mqar(N, random.Random(seed0 + t)); h += _recall_once(ids, tgt)
    return h / n
print("[capacity curve]  (base가 ~0.4 되는 N이 N*)")
for N in [2, 8, 16, 24, 32, 48, 64]:
    print(f"  N={N:>3}: {base_recall(N, n=30):.2f}")

In [ ]:
# [레버 1: 생존/Δ] target value 이후(downstream) Δ를 og<1로 → target 보존
def trials_delta(N, og, layers=None, n=30, seed0=0, scope='downstream'):
    h = 0
    for t in range(n):
        ids, pos, tgt, _ = build_mqar(N, random.Random(seed0 + t))
        L = ids.shape[1]; v0 = pos[0][1]; gv = torch.ones(L)
        if scope == 'downstream':
            gv[v0 + 1:] = og
        elif scope == 'random':
            idxs = list(range(v0 + 1, L)); random.Random(seed0 + t + 999).shuffle(idxs)
            for j in idxs[:max(1, len(idxs)//2)]: gv[j] = og
        _CTRL['gain_vec'] = gv; _CTRL['layers'] = set(layers) if layers else None
        h += _recall_once(ids, tgt)
    _CTRL['gain_vec'] = None; _CTRL['layers'] = None
    return h / n

N = 32   # 위 곡선 보고 base~0.4 되는 값으로
b = base_recall(N); print("base", round(b, 2))
print("[Gate1 dose] other-gain 스윕 (all layers)")
for og in [1.0, 0.9, 0.8, 0.7, 0.6, 0.4]:
    print(f"  og={og}: {trials_delta(N, og, n=30):.2f}")
print("[Gate1 specificity] downstream vs random (og=0.8)")
print(f"  downstream {trials_delta(N,0.8,scope='downstream',n=30):.2f} | random {trials_delta(N,0.8,scope='random',n=30):.2f}")

In [ ]:
# [국소화] 어느 레이어 그룹이 회복을 옮기나 → L16-23 기대
groups = {f"L{a}-{a+7}": list(range(a, a + 8)) for a in range(0, NL, 8)}
base = base_recall(N)
print(f"[layer localization] og=0.8, base={base:.2f}")
for name, g in groups.items():
    r = trials_delta(N, 0.8, layers=g, n=30)
    print(f"  {name}: {r:.2f}  (Δ {r-base:+.2f})")

In [ ]:
# [surgical vs full] L16-23만 vs 전층 + ppl guardrail
def ppl_under_gain(og, layers=None):
    txt = ("The transformer relies on attention while state space models compress "
           "history into a fixed hidden state for linear time inference.")
    ids = tokenizer(txt, return_tensors='pt').input_ids.to(DEVICE); L = ids.shape[1]
    _CTRL['gain_vec'] = torch.full((L,), og); _CTRL['layers'] = set(layers) if layers else None
    with torch.no_grad(): nll = model(ids, labels=ids).loss.item()
    _CTRL['gain_vec'] = None; _CTRL['layers'] = None
    return math.exp(nll)
base_ppl = ppl_under_gain(1.0); L1623 = list(range(16, 24))
print(f"base ppl {base_ppl:.1f}")
print(f"  full   og=0.8: recall {trials_delta(N,0.8,n=30):.2f}  ppl x{ppl_under_gain(0.8)/base_ppl:.2f}")
print(f"  L16-23 og=0.8: recall {trials_delta(N,0.8,layers=L1623,n=30):.2f}  ppl x{ppl_under_gain(0.8,L1623)/base_ppl:.2f}")

In [ ]:
# [write-order = 암묵 우선순위] 같은 개입, 질의 쌍 위치만 바꿈
def recall_of_pair(N, j, og, layers=None, n=30, seed0=0):
    h = 0
    for t in range(n):
        ids, pos, tgt, pairs = build_mqar(N, random.Random(seed0 + t), query_pair=j)
        L = ids.shape[1]; gv = torch.ones(L)
        if og < 1.0: gv[pos[j][1] + 1:] = og
        _CTRL['gain_vec'] = gv; _CTRL['layers'] = set(layers) if layers else None
        h += _recall_once(ids, tgt)
    _CTRL['gain_vec'] = None; _CTRL['layers'] = None
    return h / n
print("[write-order priority] N=32, L16-23 og=0.8")
for j, lab in [(0, 'early(P1)'), (N//2, 'mid(P2)'), (N-1, 'late(P3)')]:
    bb = recall_of_pair(N, j, 1.0, n=30); aa = recall_of_pair(N, j, 0.8, layers=L1623, n=30)
    print(f"  {lab:>10} pair{j:>2}: base {bb:.2f} -> {aa:.2f}  (Δ {aa-bb:+.2f})")

In [ ]:
# [graded P1/P2/P3] 문서 시그니처를 *학습 없이* graded Δ로 실현 (탐색적)
# 각 value write의 보존을 그 쌍의 레벨 og로 근사. 단조 P1>P2>P3면 graded allocation 성립.
def graded_recall(N, og_lv, layers=None, n=30, seed0=0):
    res = {1: 0, 2: 0, 3: 0}; cnt = {1: 0, 2: 0, 3: 0}
    for t in range(n):
        rng = random.Random(seed0 + t); ws = rng.sample(_WORDS, 2 * N)
        pairs = [(ws[2*i], ws[2*i+1]) for i in range(N)]
        lvl = [(i % 3) + 1 for i in range(N)]
        seq = []; pos = []
        for (k, v) in pairs:
            kp = len(seq); seq += _enc(k)
            vp = len(seq); seq += _enc(v); pos.append((kp, vp))
        for Lq in [1, 2, 3]:
            j = lvl.index(Lq)
            s2 = seq + tokenizer(" ?", add_special_tokens=False).input_ids + _enc(pairs[j][0])
            ids = torch.tensor([s2], device=DEVICE); tgt = _enc(pairs[j][1])[0]
            gv = torch.ones(ids.shape[1])
            for i, (kp, vp) in enumerate(pos):
                gv[vp] = og_lv[lvl[i]]
            _CTRL['gain_vec'] = gv; _CTRL['layers'] = set(layers) if layers else None
            res[Lq] += _recall_once(ids, tgt); cnt[Lq] += 1
    _CTRL['gain_vec'] = None; _CTRL['layers'] = None
    return {k: res[k] / cnt[k] for k in [1, 2, 3]}
g = graded_recall(N, {1: 0.7, 2: 0.85, 3: 1.0}, layers=L1623, n=30)
print("[graded P1/P2/P3] L16-23")
print(f"  P1 {g[1]:.2f} | P2 {g[2]:.2f} | P3 {g[3]:.2f}")
print("  단조 P1>P2>P3 → graded allocation 성립 / 깨지면 → SSM uniform-decay 한계(C3)")

In [ ]:
# [레버 2: 정밀도/양자화] conv1d 출력(write 표현)을 선택 위치에서 k-bit
def fake_quant_vec(v, bits):
    if bits >= 16: return v
    qmax = (2 ** bits) - 1; lo = v.amin(-1, keepdim=True); hi = v.amax(-1, keepdim=True)
    scale = (hi - lo).clamp(min=1e-8) / qmax
    return torch.round((v - lo) / scale) * scale + lo
_QMASK = {'pos': None, 'bits': 16, 'fired': 0}
def _qhook(mod, inp, out):
    if _QMASK['bits'] >= 16 or not _QMASK['pos']: return out
    L = out.shape[-1]; pos = [p for p in _QMASK['pos'] if 0 <= p < L]
    if not pos: return out
    o = out.clone()
    for p in pos: o[:, :, p] = fake_quant_vec(o[:, :, p], _QMASK['bits'])
    _QMASK['fired'] += 1; return o
_qh = [l.mixer.conv1d.register_forward_hook(_qhook) for l in model.backbone.layers]

def rd_target(bits, N, n=30, seed0=0):
    h = 0
    for t in range(n):
        ids, pos, tgt, _ = build_mqar(N, random.Random(seed0 + t))
        _QMASK['pos'] = list(pos[0]); _QMASK['bits'] = bits; h += _recall_once(ids, tgt)
    _QMASK['pos'] = None; _QMASK['bits'] = 16; return h / n
def rd_distractor(bits, N, n=30, seed0=0):
    h = 0
    for t in range(n):
        ids, pos, tgt, _ = build_mqar(N, random.Random(seed0 + t))
        dp = [p for pr in pos[1:] for p in pr]
        _QMASK['pos'] = dp; _QMASK['bits'] = bits; h += _recall_once(ids, tgt)
    _QMASK['pos'] = None; _QMASK['bits'] = 16; return h / n
print("[A] target k-bit -> recall (정밀도-왜곡 곡선)")
for bt in [16, 8, 4, 2, 1]: print(f"  {bt:>2}-bit: {rd_target(bt, N, n=30):.2f}")
print("[B] 저우선 k-bit -> target recall (중요도 배분)")
for bt in [16, 4, 2, 1]: print(f"  {bt:>2}-bit: {rd_distractor(bt, N, n=30):.2f}")

## 읽는 법 / 정직한 스코프
- **용량곡선**: base가 ~0.4 되는 N을 `N`에 넣어라(셀 7 이후).
- **Gate1**: og↓이 recall↑(특정 구간), downstream≫random이면 특이성 ✓.
- **국소화**: L16-23가 양수 피크면 회로 확인. **L16-23만으로 전층과 같은 회복 + ppl×1.00**이 핵심.
- **write-order**: early(P1) Δ가 late(P3)보다 크면 "먼저 쓴 = 암묵 우선순위".
- **graded P1/P2/P3**: 단조면 graded allocation, 깨지면 SSM uniform-decay 한계(둘 다 보고가치).
- **양자화 레버**: A 단조하락 = 정밀도 제어 성립, B target↑ = 비트 재배분 성립.

**한계(반드시 보고):** 효과는 790m 합성 MQAR에서 견고하나 **1.4b 비재현·자연어 fragile**.
이는 회로가 모델·태스크 특이적임을 뜻하며, 현상의 *경계*로 정직하게 기술한다.